# Nik Studio - rigged

**The same script.txt, animated by a rig instead of guessed at by a model.**

The AI half draws a picture and makes it move a little. This one moves a
character that is actually built - the same character every time, arms
swinging through whole arcs, and no model deciding what your line meant.

### What you need in Drive

```
My Drive / NikStudio / Mixamo /
      nik_character.fbx     the character, downloaded as a T-Pose
      idle.fbx              required
      clap.fbx  wave.fbx  jump.fbx  walk.fbx
      sway.fbx  point.fbx  crouch.fbx  nod.fbx  spin.fbx
```

All free from **mixamo.com**, free for commercial use. The file name
becomes the action name, and the action name is what a line of your
script is matched against - so rename Mixamo's "Clapping.fbx" to
"clap.fbx". That is the whole of the naming work.

`docs/RIGGED_3D.txt` has the full shopping list.

### What this does

1. Builds `Nik.blend` from those files - character, actions, three
   cameras, light, ground. Nobody opens Blender.
2. Renders one clip per line of `script.txt` into `Output/Clips`.

Then open **NikStudio_Animate.ipynb**, set `SOURCE = "clips"` and
`RUN = "video"`, and it cuts the video from them: beat cut, words on
screen, the song, the preview. There is one edit and it does not care
what made the pictures.

### No GPU needed

Workbench renders flat and fast on a CPU, which is what you want while
you are finding out whether the movements are right. Switch `ENGINE` to
`"BLENDER_EEVEE_NEXT"` and turn a GPU on when you want it to look like
something.

In [ ]:
# ======================================================================
# CELL 1 of 2 - Blender, as a Python package
# ======================================================================
#
# About 500MB and a couple of minutes. Colab will probably offer
# "RESTART SESSION" afterwards - click it. Cell 2 needs nothing from
# in here.

!pip install -q bpy imageio-ffmpeg

print("Blender installed. Now run cell 2.")

In [ ]:
# ======================================================================
# CELL 2 of 2 - build the character, render the clips
# ======================================================================

BUILD = "2026-09-13 - movements can come from a second folder"

DRIVE = "/content/drive/MyDrive"

FOLDER = DRIVE + "/NikStudio"

# Where the downloaded character is, inside FOLDER. The folder is
# searched all the way down, so an unzipped pack can go in whole.
MIXAMO = "Mixamo"

# Which character, when the pack has many. Part of the file name is
# enough - "Boy", "Astronaut". Leave it empty for the first by name.
CHARACTER = ""

# A second folder, inside FOLDER, read for movements only. Character
# packs are built for games - they ship death and punching, and a
# nursery rhyme wants clapping and waving. Put an animation library in
# here and its movements are added to the character above.
MOVEMENTS = ""

# Rebuild Nik.blend even if it is already there. Turn this on after
# adding or renaming an FBX.
REBUILD = False

# Flat and fast and no GPU, for checking that the movements are right.
# "BLENDER_EEVEE_NEXT" is the one that looks like something, and wants
# a GPU turned on.
ENGINE = "BLENDER_WORKBENCH"

WIDTH, HEIGHT = 1024, 576

# How long one clip is. The edit cuts every 2.8s and slides the clip to
# put its movement on the beat, so a second of slack is worth having.
CLIP_SECONDS = 4.0

# Only make this many, to see what they look like. 0 makes all of them.
FIRST_ONLY = 3


# ----------------------------------------------------------------------

import subprocess
import sys
from pathlib import Path

print(f"Notebook  : {BUILD}")

try:
    from google.colab import drive

    drive.mount("/content/drive")

except Exception as trouble:
    print(f"\nDrive did not mount ({type(trouble).__name__}).")

HOME = Path(FOLDER)

INTO = HOME / "Output" / "Clips"

# The two scripts this runs. Kept beside the notebook in Drive so there
# is nothing to clone and no repository to be private.
HERE = Path("/content/nikstudio")

HERE.mkdir(parents=True, exist_ok=True)

for name in ("nik_blender.py", "from_mixamo.py"):

    beside = HOME / name

    if beside.exists():
        (HERE / name).write_text(beside.read_text(encoding="utf-8"),
                                 encoding="utf-8")

if not (HERE / "from_mixamo.py").exists():
    raise SystemExit(
        f"Put nik_blender.py and from_mixamo.py in {HOME}.\n\n"
        "They are in the repository under blender/. Upload both to the "
        "NikStudio folder\nin Drive, beside Input and Output."
    )

sys.path.insert(0, str(HERE))

import from_mixamo
import nik_blender

# ------------------------------------------------------- the character

BLEND = HOME / "Nik.blend"

if REBUILD or not BLEND.exists():

    downloads = HOME / MIXAMO

    if not downloads.exists():
        raise SystemExit(
            f"No {downloads}.\n\n"
            "Download a character pack - CC0 from quaternius.com, or "
            "Mixamo - and put\nit in there. The zip can go in unzipped "
            "and whole; subfolders are read.\ndocs/AB_KYA_KARNA_HAI.txt "
            "has the four steps."
        )

    print(f"\nBuilding {BLEND.name} from {downloads.name}:")

    library = HOME / MOVEMENTS if MOVEMENTS else ""

    if library and not Path(library).exists():
        raise SystemExit(f"No {library}. MOVEMENTS names a folder "
                         f"inside {HOME}.")

    from_mixamo.build(downloads, BLEND, CHARACTER, library)

else:
    print(f"\n{BLEND.name} is already there. REBUILD = True to make it "
          f"again.")

# ------------------------------------------------------------ the shots

script = next((path for path in (HOME / "input").glob("*.txt")
               if not path.stem.lower().startswith("lyric")), None)

if script is None:
    script = next((path for path in (HOME / "Input").glob("*.txt")
                   if not path.stem.lower().startswith("lyric")), None)

if script is None:
    raise SystemExit(
        f"No script.txt in {HOME}/input.\n\n"
        "It is the same one the AI notebook uses - one scene a line."
    )

print(f"\nScript    : {script}")

lines = nik_blender.scenes_in(script)

print(f"Scenes    : {len(lines)}")

if FIRST_ONLY:

    short = HERE / "first.txt"

    short.write_text("\n".join(lines[:FIRST_ONLY]) + "\n",
                     encoding="utf-8")

    script = short

    print(f"            only the first {FIRST_ONLY} - FIRST_ONLY = 0 "
          f"for all of them")

# ---------------------------------------------------------- the render

INTO.mkdir(parents=True, exist_ok=True)

print()

nik_blender.render(
    BLEND, script, INTO,
    width=WIDTH, height=HEIGHT,
    seconds=CLIP_SECONDS, engine=ENGINE,
)

made = sorted(INTO.glob("Scene*.mp4"))

print(f"\n{len(made)} clip(s) in {INTO}")

# --------------------------------------------------- how much they move

import os


def motion_of(video):
    """The same measure the AI notebook prints, so the two compare."""

    reading = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", str(video), "-vf",
         "scale=160:90,format=gray,tblend=all_mode=difference,"
         "signalstats,metadata=print:file=-",
         "-f", "null", os.devnull],
        capture_output=True, text=True, encoding="utf-8",
        errors="replace",
    )

    scores = [float(row.rsplit("=", 1)[-1])
              for row in (reading.stdout or "").splitlines()
              if "signalstats.YAVG" in row]

    scores = scores[1:] or scores

    return sum(scores) / len(scores) if scores else 0.0


if made:

    print("\nMovement, clip by clip:")

    for clip in made[:8]:
        print(f"  {clip.stem:<10} {motion_of(clip):.2f}")

    print("\n  For comparison: the AI clips measured 3 to 5, and the "
          "finished\n  AI video 5.75. This measure counts how much of "
          "the picture changes,\n  so a plain background scores lower "
          "than a busy one - look at the\n  clips as well as the "
          "numbers.")

try:
    from IPython.display import Video, display

    if made:
        display(Video(str(made[0]), embed=True, width=640))

except Exception:
    pass

print("\n\nNext: open NikStudio_Animate.ipynb and set")
print('    SOURCE = "clips"')
print('    RUN    = "video"')
print("\nIt will cut the video from these - beat cut, words on screen, "
      "the song,\nthe preview. Nothing is generated.")